In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q2-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
import os
import pandas as pd
import numpy as np
from PIL import Image
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
import matplotlib.pyplot as plt
from torchvision.transforms.functional import to_tensor

labels_df = pd.read_csv(os.path.join(path, "labels.csv"))
img_dir = os.path.join(path, "images")

images = []
ages = []

print("Loading images...")
for index, row in labels_df.iterrows():
    img_name = row.iloc[0]
    age = row.iloc[1]
    img_path = os.path.join(img_dir, img_name)

    if os.path.exists(img_path):
        img = Image.open(img_path).convert('RGB')
        img_array = np.array(img) / 255.0
        images.append(img_array)
        ages.append(age)

X = np.array(images)
y = np.array(ages)


X = np.transpose(X, (0, 3, 1, 2))

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"y_test shape: {y_test.shape}")

In [ ]:
# 1. Convert Numpy arrays to PyTorch Tensors

import torch
from torch.utils.data import TensorDataset, DataLoader
import matplotlib.pyplot as plt
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torchvision.transforms.functional import to_tensor
from torch.optim import AdamW
from torchvision.datasets import MNIST



X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test, dtype=torch.float32)




In [ ]:
# 2. Create TensorDataset objects
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
test_dataset = TensorDataset(X_test_tensor, y_test_tensor)

In [ ]:
# 3. Create DataLoaders

batch_size = 32
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)


In [ ]:
# 4. Print shape of one batch

data_iter = iter(train_loader)
images_batch, labels_batch = next(data_iter)

print(f"Batch Image Shape: {images_batch.shape}")
print(f"Batch Label Shape: {labels_batch.shape}")

In [ ]:
# 5. Display sample images
images, labels = next(iter(train_loader))

# Display the first 6 images in the batch
plt.figure(figsize=(10, 4))
for i in range(5):
    plt.subplot(1, 5, i+1)
    # Permute the dimensions and convert to numpy for plotting
    img_to_show = images_batch[i].permute(1, 2, 0).numpy()
    plt.imshow(img_to_show)
    plt.title(f"Age: {labels_batch[i].item()}")
    plt.axis('off')
plt.tight_layout()
plt.show()



In [ ]:
# Task 1: Write your model class here:
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

# --- Task 1: Model Class ---
class NN4Layer(nn.Module):

    def __init__(self, input_dim, hidden_dim, output_dim):
        super(NN4Layer, self).__init__()

        # First linear layer: input features -> hidden layer
        self.layer1 = nn.Linear(input_dim, hidden_dim)

        # Second linear layer: hidden layer -> hidden layer
        self.layer2 = nn.Linear(hidden_dim, hidden_dim)

        # third linear layer: hidden layer -> hidden layer
        self.layer3 = nn.Linear(hidden_dim, hidden_dim)

        # Output layer: hidden layer -> number of classes (logits)
        self.layer4 = nn.Linear(hidden_dim, output_dim)

        # ReLU activation for non-linearity
        self.relu = nn.ReLU()

    # Defines how input data flows through the network
    def forward(self, x):
        # First hidden layer
        z1 = self.layer1(x)
        a1 = self.relu(z1)

        # Second hidden layer
        z2 = self.layer2(a1)
        a2 = self.relu(z2)

        # third hidden layer
        z3 = self.layer2(a2)
        a3 = self.relu(z3)



        # Output layer (raw scores / logits)
        output = self.layer3(a3) # there is something missing here, remember? :)

        return output

In [ ]:
# Task 2: Write your training loop here:
def train_step(model, train_loader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        # Forward pass
        outputs = model(images)
        loss = criterion(outputs, labels)

        # Backward pass
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    return running_loss / len(train_loader)

In [ ]:
# Task 3: Write your validation loop here:
def val_step(model, test_loader, criterion, device):
    model.eval()
    running_loss = 0.0

    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            labels = labels.view(-1, 1)

            outputs = model(images)
            loss = criterion(outputs, labels)
            running_loss += loss.item()

    return running_loss / len(test_loader)


In [ ]:
# Task 4: Define device, model, loss, optimizer:

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Initialize model with the correct input shape from our data
input_shape = X_train_tensor.shape
model = AgeRegressionModel(input_shape).to(device)

criterion = nn.MSELoss() # Mean Squared Error for regression
optimizer = optim.Adam(model.parameters(), lr=0.001)

# Calculate the total number of trainable parameters
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\nTotal trainable parameters: {total_params}")

In [ ]:
# Task 5: Start training for 20 epochs:
epochs = 20
train_losses = []
val_losses = []

for epoch in range(epochs):
      train_loss = train_step(model, train_loader, criterion, optimizer, device)
      val_loss = validate(model, criterion, test_loader, device)

      train_losses.append(train_loss)
      val_losses.append(val_loss)

      print(f'Epoch [{epoch+1}/{epochs}], Train Loss: {train_loss:.4f}, Val Loss: {val_loss:.4f}, Val Accuracy: {validate:.4f}')


In [ ]:
# Task 1: Write your code here:
plt.figure(figsize=(10, 5))
plt.plot(train_losses, label='Training Loss')
plt.plot(val_losses, label='Validation Loss')
plt.xlabel('Epochs')
plt.ylabel('Loss (MSE)')
plt.legend()
plt.title('Training vs Validation Loss')
plt.show()

In [ ]:
# Task 2 (Bonus): Write your code here: